# Notebook 6 — Model Evaluation & Comparison

**Goal:** Evaluate and compare all models using RMSE and the NASA asymmetric
scoring function. Reproduce the comparison tables from the paper.


In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from src.data_loader    import load_all_datasets, FEATURE_COLS, SENSOR_COLS
from src.preprocessor   import full_preprocess_pipeline
from src.windowing      import create_windows, create_windows_inference
from src.models.lstm_baseline import build_lstm_baseline
from src.models.lstm_dann     import build_lstm_dann
from src.train          import LSTMDANNTrainer
from src.evaluate       import rmse, nasa_score, evaluate_model, compare_models

WINDOW_SIZE = 30
MAX_RUL     = 125

datasets = load_all_datasets(data_dir='../data/raw')
for ds_id in ['FD001', 'FD002', 'FD003', 'FD004']:
    df_tr, df_te, scaler = full_preprocess_pipeline(
        df_train=datasets[ds_id]['train'],
        df_test=datasets[ds_id]['test'],
        feature_cols=FEATURE_COLS, sensor_cols=SENSOR_COLS,
        smooth=True, max_rul=MAX_RUL
    )
    datasets[ds_id]['train_norm'] = df_tr
    datasets[ds_id]['test_norm']  = df_te
    X, y, _ = create_windows(df_tr, FEATURE_COLS, WINDOW_SIZE)
    datasets[ds_id]['X_train'] = X
    datasets[ds_id]['y_train'] = y

print("Data ready.")


## 6.1 NASA Scoring Function — Asymmetric Penalty Visualisation


In [ ]:
errors = np.linspace(-50, 50, 400)
scores = np.where(
    errors < 0,
    np.exp(-errors / 13) - 1,
    np.exp( errors / 10) - 1
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(errors, scores, color='crimson', linewidth=2.5)
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.fill_between(errors[errors >= 0], 0, scores[errors >= 0],
                alpha=0.15, color='red', label='Over-prediction (heavier penalty)')
ax.fill_between(errors[errors <= 0], 0, scores[errors <= 0],
                alpha=0.12, color='blue', label='Under-prediction')
ax.set_xlabel('Prediction Error  (ŷ − y_true)', fontsize=12)
ax.set_ylabel('Score Contribution', fontsize=12)
ax.set_title('NASA Asymmetric Scoring Function\n'
             'Late predictions (positive error) are penalised more than early ones',
             fontsize=12)
ax.set_ylim(-5, 100)
ax.legend(fontsize=11)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Example penalty values
for err in [-50, -20, -10, 0, 10, 20, 50]:
    score = np.exp(-err/13)-1 if err < 0 else np.exp(err/10)-1
    print(f"  Error = {err:+3d} → Score = {score:.1f}")


**Insight:** An over-prediction of +20 cycles (claiming the engine has more
life than it does) scores ~7.4 penalty units. An under-prediction of -20
cycles scores only ~4.7. An over-prediction of +50 scores ~148 vs ~57 for
under-prediction. This asymmetry reflects real safety costs: falsely
assuring operators that equipment is healthy is far more dangerous than
scheduling preventive maintenance slightly early.


## 6.2 Load and Evaluate All Models


In [ ]:
from sklearn.model_selection import train_test_split
import os

HYPERPARAMS = {
    ('FD001', 'FD002'): dict(lstm_units=128, lstm_layers=1, feature_dim=64, reg_units=[32],  domain_units=[32],  lstm_dropout=0.5, alpha=0.8),
    ('FD001', 'FD003'): dict(lstm_units=128, lstm_layers=1, feature_dim=64, reg_units=[32],  domain_units=[32],  lstm_dropout=0.5, alpha=0.8),
    ('FD001', 'FD004'): dict(lstm_units=128, lstm_layers=1, feature_dim=64, reg_units=[32,32],domain_units=[32], lstm_dropout=0.5, alpha=1.0),
    ('FD002', 'FD001'): dict(lstm_units=64,  lstm_layers=1, feature_dim=64, reg_units=[32],  domain_units=[16,16],lstm_dropout=0.1,alpha=1.0),
    ('FD002', 'FD003'): dict(lstm_units=64,  lstm_layers=1, feature_dim=512,reg_units=[64,32],domain_units=[64,32],lstm_dropout=0.1,alpha=2.0),
    ('FD002', 'FD004'): dict(lstm_units=32,  lstm_layers=2, feature_dim=32, reg_units=[32],  domain_units=[16],  lstm_dropout=0.1, alpha=1.0),
    ('FD003', 'FD001'): dict(lstm_units=64,  lstm_layers=2, feature_dim=128,reg_units=[32,32],domain_units=[32,32],lstm_dropout=0.3,alpha=2.0),
    ('FD003', 'FD002'): dict(lstm_units=64,  lstm_layers=2, feature_dim=64, reg_units=[32,32],domain_units=[32,32],lstm_dropout=0.3,alpha=2.0),
    ('FD003', 'FD004'): dict(lstm_units=64,  lstm_layers=2, feature_dim=64, reg_units=[32,32],domain_units=[32,32],lstm_dropout=0.3,alpha=2.0),
    ('FD004', 'FD001'): dict(lstm_units=100, lstm_layers=1, feature_dim=30, reg_units=[20],  domain_units=[20],  lstm_dropout=0.5, alpha=1.0),
    ('FD004', 'FD002'): dict(lstm_units=100, lstm_layers=1, feature_dim=30, reg_units=[20],  domain_units=[20],  lstm_dropout=0.5, alpha=1.0),
    ('FD004', 'FD003'): dict(lstm_units=100, lstm_layers=1, feature_dim=30, reg_units=[20],  domain_units=[20],  lstm_dropout=0.5, alpha=1.0),
}

results_all = []

for (src, tgt), hp in HYPERPARAMS.items():
    # ── DANN inference model ───────────────────────────────────────────────
    weights_path = f'../models/saved/lstm_dann_{src}_to_{tgt}.weights.h5'
    if not os.path.exists(weights_path):
        print(f"Skipping {src}→{tgt}: weights not found (run NB05 first)")
        continue

    reg_m, dann_m = build_lstm_dann(
        window_size=WINDOW_SIZE, n_features=len(FEATURE_COLS),
        **{k: v for k, v in hp.items() if k not in ('lr_reg', 'lr_dom')}
    )
    dann_m.load_weights(weights_path)

    X_test, _ = create_windows_inference(
        datasets[tgt]['test_norm'], FEATURE_COLS, WINDOW_SIZE
    )
    y_true = datasets[tgt]['rul']

    dann_result = evaluate_model(reg_m, X_test, y_true,
                                  model_name=f'LSTM-DANN')

    # ── SOURCE-ONLY: baseline trained on SRC, applied to TGT ──────────────
    so_weights = f'../models/saved/lstm_target_only_{src}.keras'
    so_model   = build_lstm_baseline(WINDOW_SIZE, len(FEATURE_COLS))
    if os.path.exists(so_weights):
        so_model.load_weights(so_weights)
    so_result = evaluate_model(so_model, X_test, y_true,
                                model_name='SOURCE-ONLY')

    delta_pct = ((so_result['RMSE'] - dann_result['RMSE']) /
                  so_result['RMSE'] * 100)

    results_all.append({
        'Source': src, 'Target': tgt,
        'SOURCE-ONLY RMSE': so_result['RMSE'],
        'LSTM-DANN RMSE':   dann_result['RMSE'],
        'NASA (SOURCE-ONLY)': so_result['NASA_Score'],
        'NASA (DANN)':        dann_result['NASA_Score'],
        'Δ% RMSE':           round(delta_pct, 1)
    })

    print(f"{src}→{tgt}: SO={so_result['RMSE']:.2f} | DANN={dann_result['RMSE']:.2f} "
          f"| Δ={delta_pct:.1f}%")

results_df = pd.DataFrame(results_all)
print("\n", results_df.to_string(index=False))


## 6.3 RMSE Improvement Heatmap


In [ ]:
if not results_df.empty:
    pivot = results_df.pivot(index='Source', columns='Target', values='Δ% RMSE')

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        pivot, annot=True, fmt='.1f',
        cmap='RdYlGn', center=0,
        linewidths=0.5, ax=ax,
        cbar_kws={'label': 'RMSE Improvement (%)'}
    )
    ax.set_title('LSTM-DANN vs SOURCE-ONLY: % RMSE Improvement\n'
                 '(Green = DANN better, Red = DANN worse)')
    plt.tight_layout()
    plt.show()


**Reading the heatmap:**
- **Strong green cells**: DANN substantially outperforms SOURCE-ONLY.
  This typically occurs when the source domain "contains" target conditions
  (e.g., FD004 as source — it has 6 conditions and 2 fault modes, covering
  the space of all other datasets).
- **Near-zero cells**: Source and target are already similar — adaptation
  provides marginal benefit (expected result, not a failure).
- **Red cells** (rare): DANN underperformed; usually when source domain has
  fewer conditions than target (e.g., FD001→FD002), making adaptation harder.


## 6.4 RUL Prediction Timeline Comparison: Source-Only vs DANN vs Target-Only


In [ ]:
# Load TARGET-ONLY models for comparison
target_only_models = {}
for ds_id in ['FD001', 'FD002', 'FD003', 'FD004']:
    m = build_lstm_baseline(WINDOW_SIZE, len(FEATURE_COLS))
    p = f'../models/saved/lstm_target_only_{ds_id}.keras'
    if os.path.exists(p):
        m.load_weights(p)
    target_only_models[ds_id] = m

# Plot prediction timelines for a representative sample pair
PLOT_PAIRS = [
    ('FD004', 'FD001'), ('FD004', 'FD003'),
    ('FD001', 'FD003'), ('FD002', 'FD001')
]

for src, tgt in PLOT_PAIRS:
    weights_path = f'../models/saved/lstm_dann_{src}_to_{tgt}.weights.h5'
    if not os.path.exists(weights_path):
        continue

    hp = HYPERPARAMS[(src, tgt)]
    reg_m, dann_m = build_lstm_dann(
        window_size=WINDOW_SIZE, n_features=len(FEATURE_COLS),
        **{k: v for k, v in hp.items() if k not in ('lr_reg', 'lr_dom')}
    )
    dann_m.load_weights(weights_path)

    # Get test predictions for a sample engine
    df_test   = datasets[tgt]['test_norm']
    unit_id   = df_test['unit_id'].iloc[0]
    unit_data = df_test[df_test['unit_id'] == unit_id].sort_values('cycle')

    from src.windowing import create_windows_for_unit_lifecycle
    X_lc = create_windows_for_unit_lifecycle(unit_data, FEATURE_COLS, WINDOW_SIZE)
    n_windows = len(X_lc)

    y_dann = reg_m.predict(X_lc, verbose=0).flatten()

    so_model = build_lstm_baseline(WINDOW_SIZE, len(FEATURE_COLS))
    so_weights = f'../models/saved/lstm_target_only_{src}.keras'
    if os.path.exists(so_weights):
        so_model.load_weights(so_weights)
    y_so = so_model.predict(X_lc, verbose=0).flatten()

    to_model = target_only_models[tgt]
    y_to = to_model.predict(X_lc, verbose=0).flatten()

    cycles_shown = unit_data['cycle'].values[WINDOW_SIZE:]

    # True RUL for this unit from test labels
    unit_idx  = df_test['unit_id'].unique().tolist().index(unit_id)
    true_rul_end = datasets[tgt]['rul'][unit_idx]
    true_rul  = np.linspace(true_rul_end + n_windows, true_rul_end, n_windows)

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(cycles_shown, true_rul, 'k-', linewidth=2.5, label='True RUL (approx.)')
    ax.plot(cycles_shown, y_dann, 'g--', linewidth=2, label='LSTM-DANN')
    ax.plot(cycles_shown, y_so,   'r:',  linewidth=2, label='SOURCE-ONLY')
    ax.plot(cycles_shown, y_to,   'b-.',  linewidth=1.5, label='TARGET-ONLY (oracle)')
    ax.set_xlabel('Cycle')
    ax.set_ylabel('Predicted RUL')
    ax.set_title(f'RUL Prediction Timeline\n'
                 f'Source: {src} → Target: {tgt} (Engine Unit {unit_id})')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


**Insight:** The timeline plots reveal prediction quality over the full
engine lifecycle. The SOURCE-ONLY model often produces a flat or offset
curve when applied outside its training domain. LSTM-DANN tracks the
decreasing trend more closely. TARGET-ONLY (the oracle) shows the ideal
trajectory achievable with in-domain labels.
